### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [4]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [5]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\USER\AppData\Local\Temp\tmpzttklkki


In [6]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)



In [7]:
temp_dir

'C:\\Users\\USER\\AppData\\Local\\Temp\\tmp1qv2go5t'

### 2. Document Loading

In [4]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [5]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, undeunda learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural l

### Document Splitting

In [8]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # Maximum size of each chunk
    chunk_overlap=100,  # Overlap between chunks to maintain context
    length_function=len,
    separators=["\n\n","\n", "."," "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 3 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [9]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, undeunda learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    proc

### Embedding Models

In [ ]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [12]:
sample_text="MAchine LEarning is fascinating"
embeddings = OllamaEmbeddings(
    model="snowflake-arctic-embed2",
    base_url="http://localhost:11434"
)
embeddings

OllamaEmbeddings(model='snowflake-arctic-embed2', dimensions=None, validate_model_on_init=False, base_url='http://localhost:11434', client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [14]:
vector=embeddings.embed_query(sample_text)
vector

[-0.0006360473,
 -0.019540368,
 0.038615286,
 -0.021106463,
 -0.054204065,
 0.012610211,
 -0.0074212016,
 0.053116888,
 -0.060431283,
 -0.023906993,
 -0.050275996,
 -0.06547442,
 0.045317814,
 0.020392762,
 -0.053496804,
 0.026286365,
 -0.018089844,
 0.031090869,
 -0.035529416,
 -0.059703022,
 -0.045452647,
 0.040783923,
 0.024632016,
 0.032642648,
 -0.009183622,
 -0.0034949284,
 0.038551442,
 -0.07166767,
 0.030356916,
 -0.089625284,
 0.0074645635,
 0.01158363,
 0.07123145,
 0.02524205,
 -0.035488483,
 -0.076405235,
 0.044273302,
 0.0015362592,
 -0.10520278,
 0.0042737327,
 -0.0077976054,
 0.05077989,
 0.010924662,
 -0.013149573,
 -0.01076056,
 -0.032419387,
 -0.058963336,
 -0.10730323,
 -0.050129786,
 -0.054245096,
 -0.05114254,
 0.063148454,
 0.01868732,
 0.048639137,
 -0.01230301,
 -0.014169087,
 -0.008879025,
 0.025024857,
 0.0002743169,
 0.041583784,
 0.042102862,
 0.045218203,
 -0.00041960808,
 -0.034621827,
 -0.01670981,
 0.039546635,
 0.019476425,
 0.023125667,
 -0.03672547,
 

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [10]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, undeunda learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    proc

In [13]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 3 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [16]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    

In [14]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are part

In [18]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. S

In [19]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: data\doc_2.txt


### Advanced Similarity Search With Scores

In [20]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
  0.5318440198898315),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinfo

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [24]:
from langchain_ollama import ChatOllama

llm=ChatOllama(
    model="llama3.2:latest",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=1000,
    num_ctx=2048
)


In [25]:
test_response=llm.invoke("What is Large Language Models")
test_response

AIMessage(content='Large Language Models (LLMs) are a type of artificial intelligence (AI) model that uses natural language processing (NLP) to understand, generate, and process human language. They are designed to learn patterns in large amounts of text data, allowing them to make predictions or take actions based on the input they receive.\n\nCharacteristics of Large Language Models:\n\n1. **Scale**: LLMs are trained on massive amounts of text data, often in the tens of billions of parameters.\n2. **Complexity**: They use complex algorithms and techniques, such as transformer architectures, to process and generate human language.\n3. **Contextual understanding**: LLMs can understand context and nuances of language, allowing them to make more accurate predictions or take more informed actions.\n4. **Generative capabilities**: Many LLMs have the ability to generate text, images, or other forms of content based on input prompts.\n\nTypes of Large Language Models:\n\n1. **Transformer-bas

In [26]:
for chunk in llm.stream("What is Large Language Models?"):
    print(chunk.content, end="", flush=True)

Large Language Models (LLMs) are a type of artificial intelligence (AI) model that uses natural language processing (NLP) to understand, generate, and process human language. They are designed to learn patterns in large amounts of text data, allowing them to make predictions or take actions based on the input they receive.

The key characteristics of Large Language Models include:

1. **Scale**: LLMs are trained on massive amounts of text data, often in the tens of billions of parameters.
2. **Complexity**: They use complex neural network architectures, such as transformer models, to process and generate language.
3. **Contextual understanding**: LLMs can understand the context of a sentence or paragraph, including nuances like idioms, sarcasm, and figurative language.

LLMs have many applications, including:

1. **Language translation**: They can translate text from one language to another with high accuracy.
2. **Text summarization**: They can summarize long pieces of text into conci

In [15]:
from langchain.chat_models.base import init_chat_model

llm=init_chat_model("ollama:qwen3.5:4b", 
    base_url="http://10.11.200.109:11434",
    temperature=0,
    num_predict=2000,
    num_ctx=2048)
#llm=init_chat_model("groq:")
llm

ChatOllama(output_version=None, model='qwen3.5:4b', num_ctx=2048, num_predict=2000, temperature=0.0, base_url='http://10.11.200.109:11434')

In [16]:
for chunk in llm.stream("What is AI?"):
    print(chunk.content, end="", flush=True)

At its simplest level, **Artificial Intelligence (AI)** is a branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. These tasks include learning from data, reasoning, problem-solving, perception (like vision or speech), and understanding natural language.

Here is a breakdown of what AI actually entails:

### 1. How It Works
AI doesn't "think" like humans; it processes information based on patterns. The core engine behind most modern AI is **Machine Learning**. Instead of being explicitly programmed with rules for every possible scenario, machine learning algorithms are fed vast amounts of data and allowed to find correlations within that data to make predictions or decisions.

### 2. Types of AI
It is important to distinguish between what exists today and the theoretical future:

*   **Narrow (or Weak) AI:** This is the type we use right now. It excels at specific tasks but cannot generalize them outside its prog

### Modern RAG Chain

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [18]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000015C0FD38410>, search_kwargs={})

In [19]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate

system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [20]:
prompt.pretty_print()

================================ System Message ================================

You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---

================================ Human Message =================================

{input}


##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [21]:
### Create a document chain (LCEL approach - not needed, we build full chain next)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [22]:
from langchain_core.runnables import RunnableLambda

### Create The Final RAG Chain (LCEL)
# Fix: Convert ChatPromptValue to messages for Ollama LLM
def prompt_to_messages(prompt_value):
    """Convert ChatPromptValue to list of messages"""
    if hasattr(prompt_value, 'to_messages'):
        return prompt_value.to_messages()
    return prompt_value

rag_chain = (
    {
        "context": RunnableLambda(lambda x: x["input"]) | retriever | format_docs,
        "input": RunnableLambda(lambda x: x["input"])
    }
    | prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)

In [23]:
rag_chain

{
  context: RunnableLambda(...)
           | VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000015C0FD38410>, search_kwargs={})
           | RunnableLambda(format_docs),
  input: RunnableLambda(...)
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nRetrieved context:\n{context}\n\n---"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| RunnableL

In [24]:
response = rag_chain.invoke({"input": "What is Machine Learning?"})
print("RAG Response:")
print(response)

RAG Response:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It encompasses three main types, including supervised, unsupervised, and reinforcement learning. These methods allow models to process data differently depending on the specific task requirements.


In [25]:
# Function to query the RAG system
def query_rag(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Get answer using LCEL chain
    answer = rag_chain.invoke({"input": question})
    print(f"Answer: {answer}")
    
    # Also show source documents
    docs = retriever.invoke(question)
    print("\nRetrieved Context:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return answer

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: There are three main types of machine learning: supervised, unsupervised, and reinforcement learning. Supervised models use labeled data to train, while unsupervised methods identify patterns in unlabeled datasets. Finally, reinforcement learning improves through interaction with an environment using rewards and penalties.

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on th

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [26]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [27]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant that provides answers based on the given context."),
    ("human", """Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
])
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an assistant that provides answers based on the given context.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [28]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000015C0FD38410>, search_kwargs={})

In [29]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [30]:
## Build the chain using LCEL
rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)

In [31]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Based on the provided context, Deep Learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Additionally, deep learning has revolutionized fields like computer vision, natural language processing, and speech recognition.'

In [32]:
retriever.invoke("What is Deep Learning")

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, undeunda learning, and reinforcement \n    learning. Super

In [33]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [34]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: Based on the provided context in the "Machine Learning Fundamentals" section, the key concept for reinforcement learning is that it **learns through interaction with an environment using rewards and penalties**. It is described as one of three main types of machine learning (alongside supervised and unsupervised learning).

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Natural Language Processing (NLP)

    NLP is a fiel

In [35]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: Based on the provided context, machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. 

The text further specifies three main types within this field:
*   **Supervised learning**, which uses labeled data to train models.
*   **Unsupervised learning**, which finds patterns in unlabeled data.
*   **Reinforcement learning**, which learns through interaction with an environment using rewards and penalties.

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks ar

In [36]:
query_rag_lcel("What is depe learning?")

Question: What is depe learning?
--------------------------------------------------
Answer: Based on the context provided, **deep learning** is defined as:

*   A subset of machine learning based on artificial neural networks.
*   Networks that are inspired by the human brain and consist of layers of interconnected nodes.
*   Technology that has revolutionized fields like computer vision, natural language processing, and speech recognition.

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses o

### Add New Documents To Existing Vector Store

In [37]:
vectorstore

In [38]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [39]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [40]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, undeunda learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    proc

In [41]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [42]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [43]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [44]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)

['8ac8c11d-77b7-4456-b178-55309f21d9e3']

In [45]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 1 new chunks to the vector store
Total vectors now: 4


In [46]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: Based on the context provided, the key concepts in reinforcement learning are **states**, **actions**, **rewards**, **policies**, and **value functions**. 

This information is explicitly stated in the "Reinforcement Learning in Detail" section of the text.

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networ

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [47]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [48]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
llm = init_chat_model("ollama:llama3.2:3b", 
    base_url="http://10.11.200.109:11434",
    temperature=0,
    num_predict=2000,
    num_ctx=2048)

In [56]:
## create history aware retriever (with qwen3.5:4b - has chat history bug)
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# NOTE: This uses qwen3.5:4b which has a bug with MessagesPlaceholder + chat_history
# Use llama3.2 version below for working conversational RAG

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000015C0FD38410>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIM

In [57]:
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
    ])

qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_answer_chain)

In [58]:
qa_prompt.pretty_print()

================================ System Message ================================

You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}


In [59]:
chat_history=[]
result1 = rag_chain.invoke({
    "input": "What is machine learning?",
    "chat_history": chat_history  # This should be a list of messages representing the conversation history
})

print("RAG Response:")
print(result1['answer'])

RAG Response:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It uses data to train models, allowing them to make predictions or take actions on their own. Machine learning has three main types: supervised learning, unsupervised learning, and reinforcement learning.


In [60]:
chat_history.extend([
    HumanMessage(content="What is machine learning?"),
    AIMessage(content=result1['answer'])
])

In [61]:
result2 = rag_chain.invoke({
     "chat_history": chat_history,
    "input": "What are its main types?"
})
print("RAG Response:")
print(result2['answer'])

RAG Response:
Machine learning's main types are:

1. Supervised learning
2. Unsupervised learning
3. Reinforcement learning.

These categories help determine how the machine learning model learns from data or interacts with an environment.


In [62]:
chat_history.extend([
    HumanMessage(content="What are its main types?"),
    AIMessage(content=result2['answer'])
])

In [63]:
chat_history

[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It uses data to train models, allowing them to make predictions or take actions on their own. Machine learning has three main types: supervised learning, unsupervised learning, and reinforcement learning.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What are its main types?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Machine learning's main types are:\n\n1. Supervised learning\n2. Unsupervised learning\n3. Reinforcement learning.\n\nThese categories help determine how the machine learning model learns from data or interacts with an environment.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [64]:
result3 = rag_chain.invoke({
    "input": "Tell me more.",
    "chat_history": chat_history
})
print("RAG Response:")
print(result3['answer'])

RAG Response:
I don't know more about the specific details of each type, but I can tell you that:

* Supervised learning uses labeled data to train models
* Unsupervised learning finds patterns in unlabeled data
* Reinforcement learning learns through interaction with an environment using rewards and penalties.

If you'd like to know more about any of these types, feel free to ask!


### 📊 Model Comparison Results & Root Cause Analysis

#### Test Results: qwen3.5:4b vs llama3.2

| Model | Empty Chat History | With Chat History | "Tell me more" Follow-up |
|-------|-------------------|-------------------|--------------------------|
| **qwen3.5:4b** | ✅ Works (138 chars) | ❌ Returns EMPTY | ❌ Returns EMPTY |
| **llama3.2** | ✅ Works (145 chars) | ✅ Works (374 chars) | ✅ Works (214 chars) |

---

#### 🔍 Why llama3.2 Works but qwen3.5:4b Fails

**Root Cause: Model-Specific Prompt Handling Bug in qwen3.5:4b**

The issue is NOT with LangChain's `create_retrieval_chain` or `MessagesPlaceholder` - both work correctly. The problem is **how different Ollama models parse prompts with chat history**.

**What Happens Under the Hood:**

1. **`MessagesPlaceholder("chat_history")`** in the prompt template creates a slot for conversation history
2. When chat history is populated, it inserts multiple `HumanMessage` and `AIMessage` objects
3. These get converted to Ollama's message format: `[{"role": "user", ...}, {"role": "assistant", ...}, ...]`
4. **qwen3.5:4b** appears to have a tokenization or parsing bug when processing:
   - System message + Multiple chat history messages + Current user message
   - This specific combination causes the model to return empty output
   
5. **llama3.2** handles this message structure correctly and generates proper responses

**Technical Details:**

```python
# This prompt structure causes qwen3.5:4b to fail:
ChatPromptTemplate.from_messages([
    ("system", "You are an assistant..."),
    MessagesPlaceholder("chat_history"),  # <-- When this has >0 messages
    ("human", "{input}")
])

# With chat_history = [HumanMessage(...), AIMessage(...), ...]
# qwen3.5:4b returns: "" (empty string)
# llama3.2 returns: Full, contextual answer ✓
```

**Why This Matters for Conversational RAG:**

- **Query Reformulation**: `create_history_aware_retriever` needs chat history to rephrase ambiguous questions
  - "Tell me more" → "Tell me more about machine learning types"
- Without working chat history, the retriever can't understand context
- Results in poor document retrieval and irrelevant answers

---

#### ✅ Recommended Solution

**Use llama3.2 (or other compatible models) for conversational RAG:**

```python
llm = ChatOllama(
    model="llama3.2",  # Works with MessagesPlaceholder
    base_url="http://localhost:11434",
    temperature=0
)
```

**Models to Avoid for Conversational RAG:**
- qwen3.5:4b (has chat history bug)

**Alternative Models to Try:**
- llama3.2 ✅ (tested, works)
- llama3.1 (likely works)
- mistral (likely works)
- Other Llama-based models

---

#### Key Takeaway

The `create_retrieval_chain` pattern with `create_history_aware_retriever` is the **correct and recommended approach** for conversational RAG in modern LangChain. The empty response issue was purely a model compatibility problem, not a framework issue.

### 🎯 Summary: Best Practices for Conversational RAG

#### Recommended Pattern (Modern LangChain)

```python
# 1. Create prompt for query reformulation
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reformulate question to be standalone..."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# 2. Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# 3. Create QA prompt with chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context..."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# 4. Create document chain
qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 5. Combine everything
rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    qa_answer_chain
)

# 6. Use with chat history
result = rag_chain.invoke({
    "input": "user question",
    "chat_history": [HumanMessage(...), AIMessage(...)]
})
```

#### Why This Approach is Better Than Manual LCEL

| Feature | `create_retrieval_chain` | Manual LCEL |
|---------|--------------------------|-------------|
| **Query Reformulation** | ✅ Built-in via `history_aware_retriever` | ❌ Must implement manually |
| **Code Length** | Shorter, cleaner | Longer, more verbose |
| **Maintenance** | Easier to update | More complex |
| **Context Handling** | Automatic | Manual extraction needed |
| **Production Ready** | Yes | Requires more testing |

#### Handling Ambiguous Follow-ups

The `history_aware_retriever` automatically reformulates ambiguous questions:

```
User: "Tell me more"
↓
LLM reformulates: "Tell me more about the types of machine learning"
↓
Retriever searches: "types of machine learning"
↓
Returns relevant documents ✓
```

Without this, the retriever would search for literal "tell me more" → wrong documents → wrong answer.

##### LCEL based approach

In [ ]:
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# NOTE: Using qwen3.5:4b - will fail with chat history
qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_answer_chain)

print("⚠️  Chain created with qwen3.5:4b (has known issues with chat history)")
print("   For working conversational RAG, see llama3.2 implementation below")

# Build conversational chain with LCEL
# Important: Extract the input question before passing to retriever
conversational_rag_chain = (
    {
        "context": RunnableLambda(lambda x: x["input"]) | retriever | format_docs,
        "chat_history": RunnableLambda(lambda x: x.get("chat_history", [])),
        "input": RunnableLambda(lambda x: x["input"])
    }
    | qa_prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)
print("Conversational RAG chain created!")

In [ ]:
chat_history = []
# First question
answer1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {answer1}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=answer1)
])

In [ ]:
chat_history

In [ ]:
# Follow-up question
answer2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"
})
print(f"Q: What are its main types?")
print(f"A: {answer2}")

In [ ]:
# Answer already printed above

### Using GROQ LLM's

**Note:** If `GROQ_API_KEY` is not loading:
1. Make sure your `.env` file contains: `GROQ_API_KEY=your_actual_key_here`
2. The `.env` file should be in the root directory: `c:\RAG\.env`
3. Run the cell below with `load_dotenv(override=True)` to force reload
4. If you added the key after starting the notebook, you MUST use `override=True`

In [ ]:
# Force reload .env file (in case it was modified after notebook startup)
load_dotenv(override=True)

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print("✓ GROQ_API_KEY loaded successfully")
else:
    print("✗ GROQ_API_KEY not found in .env file")
    print("  Make sure your .env file contains: GROQ_API_KEY=your_key_here")
# Note: Don't print groq_key to avoid exposing it in outputs

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [ ]:
# Set GROQ_API_KEY in environment
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    print("✓ GROQ_API_KEY set in environment")
else:
    print("⚠️  GROQ_API_KEY not found - please check your .env file")

In [ ]:
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=os.getenv("GROQ_API_KEY"))
llm

In [ ]:
llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm

In [ ]:
llm.invoke("What is AI?")